# Solución 2: Recomendación Colaborativa Ítem-Ítem (Similitud Coseno por Coocurrencia)
* **Objetivo de Negocio**: Sugerir productos afines en el carrito de compras para aumentar el número de unidades vendidas por pedido (*Cross-Selling*).
* **Unidad de Análisis**: Una interacción de presencia de un producto dentro de un pedido entregado.
* **Técnica Analítica**: Filtrado Colaborativo basado en Ítems utilizando Similitud Coseno por Coocurrencia.
* **Origen de Datos**: Repositorio GitHub en línea (`pryBinaBack/outputs/dataset_recomendacion_interacciones.csv`).


## 1. Dependencias e importación de librerías


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print


## 2. Carga de Datasets (En línea desde GitHub con Respaldo Local)


In [ ]:
url_interacciones = "https://raw.githubusercontent.com/Oscar-David-Dela-Cruz-Hdez/pryBinaBack/master/outputs/dataset_recomendacion_interacciones.csv"
url_productos = "https://raw.githubusercontent.com/Oscar-David-Dela-Cruz-Hdez/pryBinaBack/master/outputs/dataset_productos.csv"

print(f"Intentando cargar datasets en línea desde GitHub...")
df_interacciones = pd.read_csv(url_interacciones)
df_productos = pd.read_csv(url_productos)
print("Datasets de recomendación cargados con éxito directamente desde GitHub.")

print("Interacciones cargadas:", len(df_interacciones))
print("Productos cargados:", len(df_productos))
display(df_interacciones.head())


## 3. Matriz de Coocurrencia y Construcción del Recomendador Coseno


In [ ]:
# Agrupar productos por pedido
canastas = df_interacciones.groupby("pedido_id")["producto_id"].apply(lambda s: list(set(s.dropna()))).values

frecuencia_individual = {}
coocurrencia = {}

for canasta in canastas:
    for prod in canasta:
        frecuencia_individual[prod] = frecuencia_individual.get(prod, 0) + 1
    for i in range(len(canasta)):
        for j in range(i + 1, len(canasta)):
            par = tuple(sorted([canasta[i], canasta[j]]))
            coocurrencia[par] = coocurrencia.get(par, 0) + 1

print(f"Canastas analizadas: {len(canastas)}")
print(f"Productos distintos: {len(frecuencia_individual)}")
print(f"Pares con coocurrencia: {len(coocurrencia)}")

def calcular_similitud(prod_a, prod_b):
    par = tuple(sorted([prod_a, prod_b]))
    juntos = coocurrencia.get(par, 0)
    if juntos == 0:
        return 0.0
    freq_a = frecuencia_individual.get(prod_a, 0)
    freq_b = frecuencia_individual.get(prod_b, 0)
    if freq_a == 0 or freq_b == 0:
        return 0.0
    return juntos / np.sqrt(freq_a * freq_b)

def recomendar_para_carrito(productos_carrito, limite=6):
    semillas = set(productos_carrito)
    puntuaciones = {}
    
    for semilla in semillas:
        for candidato in frecuencia_individual:
            if candidato in semillas:
                continue
            sim = calcular_similitud(semilla, candidato)
            if sim > 0:
                puntuaciones[candidato] = puntuaciones.get(candidato, 0.0) + sim
                
    ordenados = sorted(puntuaciones.items(), key=lambda x: x[1], reverse=True)[:limite]
    
    res = []
    for prod_id, puntaje in ordenados:
        info = df_productos[df_productos["producto_id"] == prod_id]
        nombre = info["nombre"].values[0] if len(info) else f"Producto {prod_id}"
        precio = info["precioNormal"].values[0] if len(info) else 0
        res.append({
            "producto_id": prod_id,
            "nombre": nombre,
            "precioNormal": precio,
            "puntuacion": round(puntaje, 4)
        })
    return pd.DataFrame(res)


## 4. Prueba de Recomendación de Negocio y Visualización (Mapa de Calor)


In [ ]:
# Simular un carrito con un producto semilla
primer_producto_id = list(frecuencia_individual.keys())[0]
info_semilla = df_productos[df_productos["producto_id"] == primer_producto_id]
nombre_semilla = info_semilla["nombre"].values[0] if len(info_semilla) else primer_producto_id

print(f"[CARRITO] Carrito del cliente contiene: {nombre_semilla}")
print("\n=== RECOMENDACIONES GENERADAS POR SIMILITUD COSENO ===")
df_rec = recomendar_para_carrito([primer_producto_id], limite=6)
display(df_rec)

# Construir submatriz de similitud para heatmap de 15 productos
top_prods = list(frecuencia_individual.keys())[:15]
matriz_sim = np.zeros((len(top_prods), len(top_prods)))

for i, p1 in enumerate(top_prods):
    for j, p2 in enumerate(top_prods):
        if i == j:
            matriz_sim[i, j] = 1.0
        else:
            matriz_sim[i, j] = calcular_similitud(p1, p2)

plt.figure(figsize=(10, 8))
sns.heatmap(matriz_sim, xticklabels=top_prods, yticklabels=top_prods, cmap="YlGnBu", annot=False)
plt.title("Similitud Coseno por Coocurrencia entre Productos (Submatriz 15x15)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()
